# Pack a repo into QR sheets

`qrpack.py` as a notebook, with nothing to import. Run it on the **offline**
machine: paste the cells, point `ROOT` at your repo, run, photograph the screen.

**Nothing is written to disk.** No images, no HTML files, no temp files. Every
sheet is an SVG string that exists only in the cell output until you photograph
it, and closing the notebook without saving leaves no trace.

Needs `segno` (`pip install segno`). Rebuild on the receiving machine with
`agunpack.py`.

## 1. What gets sent

In [1]:
import base64, hashlib, io, json, lzma, os, struct, tarfile, zlib
from collections.abc import Sequence

import segno
from IPython.display import HTML, display

StrPath = str | os.PathLike[str]

DEFAULT = (".py", ".ipynb", ".zsh", ".md", ".bash", ".json", ".sh", ".toml", ".code-workspace")

# DEFAULT decides what is sent; these two only stop it reaching somewhere silly.
# Both earn their line: .mypy_cache alone is 145 .json files (a .venv would be
# thousands of .py), and vault/source_tokens.zsh is a credential that .zsh would
# otherwise put on a screen you are about to photograph.
SKIP = {".git", "node_modules", "__pycache__", ".venv", "venv", "build", "dist", ".mypy_cache", ".pytest_cache"}
SECRET = ("secret", "token", "password", "credential", ".env", "vault/", ".pem", ".key", "id_rsa", ".netrc")

## 2. Reading a file

Notebooks are sent as **input cells only** — no outputs, no execution counts, no
metadata. On a real tree that is the difference between ~24 photographs and ~848,
and stored output is the one part you regenerate for free by re-running.

`outputs` and `execution_count` survive as empty/null on code cells purely
because the nbformat v4 schema requires the keys.

In [2]:
def read(path: StrPath, rel: str) -> bytes:
    """Bytes as sent: notebooks stripped to input cells, everything else raw."""
    with open(path, "rb") as fh:
        raw = fh.read()
    if not rel.endswith(".ipynb"):
        return raw
    try:
        nb = json.loads(raw)
    except (ValueError, UnicodeDecodeError):
        return raw
    cells: list[dict] = []
    for c in nb.get("cells", []):
        if (k := c.get("cell_type")) not in ("code", "markdown", "raw"):
            continue
        s = c.get("source", "")
        cells.append({"cell_type": k, "metadata": {}, "source": s.splitlines(True) if isinstance(s, str) else s})
        if k == "code":
            cells[-1].update(execution_count=None, outputs=[])
    # minor 4 deliberately: 4.5 requires a per-cell "id" and we emit none, so
    # declaring 5 would produce notebooks that are invalid by their own schema.
    doc = {"cells": cells, "metadata": {}, "nbformat": nb.get("nbformat", 4), "nbformat_minor": 4}
    return (json.dumps(doc, indent=1, ensure_ascii=False) + "\n").encode()

## 3. Choosing the files

In [3]:
def collect(root: StrPath, include: Sequence[str] = DEFAULT, max_size: int = 1 << 20) -> list[str]:
    """Relative paths pack() would send, cheapest checks first."""
    root = os.path.abspath(os.path.expanduser(root))
    exts, out = tuple(include), []
    for dp, dn, fn in os.walk(root):
        dn[:] = sorted(d for d in dn if d not in SKIP)
        for n in sorted(fn):
            full = os.path.join(dp, n)
            rel = os.path.relpath(full, root).replace(os.sep, "/")
            if not n.endswith(exts) or os.path.islink(full) or not os.path.isfile(full):
                continue
            if any(s in rel.lower() for s in SECRET):
                continue
            blob = read(full, rel)  # notebooks are measured after stripping
            if len(blob) > max_size or b"\0" in blob[:8192]:
                continue
            out.append(rel)
    return out

## 4. Packing

tar → xz → base32 → QR frames → sheets of HTML.

Defaults are tuned for a **3840x2160** sending display: a 3500x1834 sheet
carrying 21.1 KB, with ~340x326 px of headroom. For a smaller screen put less
on a sheet — 1440p: `chunk=1350, cols=4, rows=2`; 1080p: `chunk=750, cols=4,
rows=2`.

`scale` is **CSS** pixels per QR module, the number that decides whether a photo
can be decoded at all; below ~3 the display never renders the detail and no
camera recovers it. Two things about it:

- **Keep it a whole number.** At 4.5 every module edge lands on a half pixel and
  the browser antialiases it — exactly the softness that costs you a decode.
- **Halve it on a HiDPI desktop.** On macOS at 2x a 4K panel reports a 1920x1080
  CSS viewport, so `scale=2` there is still 4 *device* pixels per module. Run the
  next cell to find out which case you are in.

In [4]:
def pack(
    root: StrPath,
    include: Sequence[str] = DEFAULT,
    chunk: int = 1200,
    cols: int = 6,
    rows: int = 3,
    scale: int = 4,
    max_size: int = 1 << 20,
    quiet: bool = False,
) -> dict[int, str]:
    """Return {sheet_number: html_fragment} - one entry per photograph.

    Writes nothing anywhere. Each fragment carries its own inline styles and a
    white background, so it renders correctly in a dark-theme notebook.

    Defaults suit a 3840x2160 sending display: a 3500x1834 sheet carrying
    21.1 KB, leaving ~340x326 px of headroom. 1440p: chunk=1350, cols=4,
    rows=2. 1080p: chunk=750, cols=4, rows=2.
    """
    root = os.path.abspath(os.path.expanduser(root))
    files = collect(root, include, max_size)
    if not files:
        raise ValueError(f"no files matched under {root}")

    entries: list[dict[str, str | int]] = []
    buf = io.BytesIO()
    with tarfile.open(fileobj=buf, mode="w", format=tarfile.PAX_FORMAT) as tar:
        for rel in [*files, None]:  # None is the manifest, so it lands complete
            if rel is None:
                name = ".agpack-manifest.json"
                blob = json.dumps({"format": "agpack/1", "files": entries}, sort_keys=True).encode()
                mode = 0o644
            else:
                full = os.path.join(root, rel)
                name, blob = rel, read(full, rel)
                mode = 0o755 if os.stat(full).st_mode & 0o111 else 0o644
                entries.append({"path": rel, "bytes": len(blob), "sha256": hashlib.sha256(blob).hexdigest()})
            info = tarfile.TarInfo(name)
            info.size, info.mtime, info.uid, info.gid, info.mode = len(blob), 0, 0, 0, mode
            tar.addfile(info, io.BytesIO(blob))

    payload = lzma.compress(buf.getvalue(), preset=9)
    sha = hashlib.sha256(payload).hexdigest()
    tid = int(sha[:8], 16)
    parts = [payload[i : i + chunk] for i in range(0, len(payload), chunk)]

    def frame(seq: int, blob: bytes) -> str:
        head = struct.pack(">2sBIHHI", b"AG", 1, tid, seq, len(parts), zlib.crc32(blob) & 0xFFFFFFFF)
        return base64.b32encode(head + blob).decode().rstrip("=")

    header = json.dumps(
        {
            "format": "agpack/1",
            "transfer_id": "%08x" % tid,
            "sha256": sha,
            "compressed_bytes": len(payload),
            "codec": "tar+xz+base32",
            "chunk_bytes": chunk,
            "grid": "%dx%d" % (cols, rows),
        },
        sort_keys=True,
        separators=(",", ":"),
    ).encode()
    frames = [(0, frame(0, header))] + [(i, frame(i, c)) for i, c in enumerate(parts, 1)]

    per = cols * rows
    total = -(-len(frames) // per)
    sheets = {}
    for s in range(0, len(frames), per):
        cells = "".join(
            "<div style='text-align:center'>%s<div style='font:700 12px monospace;color:#000'>%d</div></div>"
            % (segno.make(t, error="m", mode="alphanumeric").svg_inline(scale=scale, border=3), seq)
            for seq, t in frames[s : s + per]
        )
        sheets[s // per + 1] = (
            "<div style='background:#fff;padding:14px;display:inline-block'>"
            "<div style='font:700 14px monospace;color:#000;padding-bottom:8px'>"
            "SHEET %d / %d &nbsp;&middot;&nbsp; id %08x &nbsp;&middot;&nbsp; frames %d-%d</div>"
            "<div style='display:grid;grid-template-columns:repeat(%d,auto);gap:8px'>%s</div></div>"
            % (s // per + 1, total, tid, frames[s][0], frames[min(s + per, len(frames)) - 1][0], cols, cells)
        )

    if not quiet:
        raw = sum(int(e["bytes"]) for e in entries)  # as sent, not as on disk
        print(
            f"{len(files)} files, {raw/1024:,.0f} KB -> {len(payload)/1024:,.0f} KB "
            f"compressed -> {len(frames)} frames -> {total} photographs"
        )
        print(f"transfer id {tid:08x}   sha256 {sha[:32]}...")
        print("nothing written to disk")
    return sheets

## 4b. Which scale does your display want?

In [5]:
display(
    HTML(
        "<div style='font:600 13px system-ui;padding:8px;background:#fff;color:#000'>"
        "<span id='dpr'></span></div>"
        "<script>document.getElementById('dpr').textContent="
        "'CSS viewport ' + window.innerWidth + 'x' + window.innerHeight +"
        "'  |  devicePixelRatio ' + window.devicePixelRatio +"
        "'  |  use scale=' + (window.devicePixelRatio >= 2 ? 2 : 4);</script>"
    )
)

## 5. Check before you shoot

Look at the file list first — this is the last point where a surprise is cheap.

In [ ]:
ROOT = "../.."  # the repo to send

files = collect(ROOT)
print(f"{len(files)} files\n")
for f in files[:8]:
    print("  ", f)
print(f"   ... and {len(files) - 8} more")

295 files

   README.md
   claude.code-workspace
   kretsinger-lite.code-workspace
   kretsinger.code-workspace
   pyproject.toml
   sitecustomize.py
   tex-fmt.toml
   .vscode/settings.json
   ... and 287 more


## 6. Pack

In [7]:
sheets = pack(ROOT)

295 files, 1,077 KB -> 246 KB compressed -> 212 frames -> 12 photographs
transfer id 1a34e482   sha256 1a34e482dd8c9262d3db4c799182cdf2...
nothing written to disk


## 7. Photograph

Inline preview is for confirming a sheet looks right. To actually photograph,
use the links below: each opens one sheet in a new browser tab as a `data:` URL —
still nothing on disk — where you can fullscreen it so the symbols get the screen
pixels they need. The notebook output area is usually too narrow to show a whole
sheet at 1:1, and anything that scales the symbols down costs you decodability.

In [ ]:
display(HTML(sheets[1]))  # preview sheet 1 inline

In [9]:
def tab_link(html: str, n: int) -> str:
    """A data: URL so a sheet opens fullscreen without ever touching disk."""
    doc = (
        "<html><body style='margin:0;background:#fff;display:flex;"
        f"align-items:center;justify-content:center;height:100vh'>{html}</body></html>"
    )
    b64 = base64.b64encode(doc.encode()).decode()
    return f"<a href='data:text/html;base64,{b64}' target='_blank' style='font:600 13px system-ui;padding-right:14px'>sheet {n}</a>"


display(
    HTML(
        "<div style='padding-top:12px;font:600 13px system-ui'>open fullscreen to photograph:</div>"
        + "".join(tab_link(sheets[n], n) for n in sorted(sheets))
    )
)

## 8. Then

One photo per sheet: square-on, filling the frame, tapped to focus, no glare.
Order does not matter and duplicates are harmless — every frame carries its own
index and CRC. Check the first photo is sharp before shooting the rest.

On the receiving machine:

```bash
python3 agunpack.py decode ~/photos/transfer --out ~/recovered
```

Fewer photographs: `pack(ROOT, chunk=1100)` on a 1440p display, plus
`cols=6, rows=3` on 4K. Throughput scales hard with the sending screen — if the
offline machine can drive an external monitor, use the biggest one you can find.

To send something other than the defaults, pass extensions directly:
`pack(ROOT, include=(".py", ".md"))`.

**Before saving this notebook**, clear its output — the sheets are large SVG
blobs, and a saved copy is the one way this process leaves your repo on disk.